# Klassifikation mit allen Werten

`JobSat` als Zielwert


In [7]:
import os, glob
print("cwd:", os.getcwd())
print("local xgboost candidates:", glob.glob("xgboost*"))


cwd: /Users/jonas/Documents/DataAnalytics/Project
local xgboost candidates: []


In [3]:
import sys
!{sys.executable} -m pip -V
!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install --no-cache-dir -U xgboost


pip 25.3 from /Users/jonas/Documents/DataAnalytics/Project/.venv/lib/python3.13/site-packages/pip (python 3.13)
Found existing installation: xgboost 3.1.2
Uninstalling xgboost-3.1.2:
  Successfully uninstalled xgboost-3.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 5.3 MB/s  0:00:00 eta 0:00:01


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [10]:

df = pd.read_csv("One-Hot-Encoded.csv")


bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

MaxAge                                                            float64
AgeNum                                                            float64
WorkExp                                                           float64
YearsCode                                                         float64
RemoteCategoryNum                                                 float64
                                                                   ...   
AIAgents_no, i use ai exclusively in copilot/autocomplete mode      int64
AIAgents_yes, i use ai agents at work daily                         int64
AIAgents_yes, i use ai agents at work monthly or infrequently       int64
AIAgents_yes, i use ai agents at work weekly                        int64
AIAgents_nan                                                        int64
Length: 484, dtype: object

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [13]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return 0
    elif x <= 6:
        return 1
    else:
        return 2


def map_remote(x):
    x = float(x)
    if x <= 0:
        return 0
    elif x <= 0.25:
        return 1
    elif x <= 0.5:
        return 2
    elif x <= 0.75:
        return 3
    else:
        return 4
     


## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`



In [14]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"] #?

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

#y = df["JobSat"].apply(map_jobsat)
y = df["RemoteCategoryNum"].apply(map_remote)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()


X.head()

Textspalten: ['LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIAgent_Uses']
Numerische Spalten: ['MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'CompTotal', 'ConvertedCompYearly', 'ConvertedCompTotal', 'MainBranch_i am a developer by profession', 'MainBranch_i am learning to code', 'MainBranch_i am not primarily a developer, but i write code sometimes as part of my work/studies', 'MainBranch_i code primarily as a hobby', 'MainBranch_i used to be a developer by profession, but no longer am', 'MainBranch_i work with developers or my work supports developers but am not a developer by profession', 'MainBranch_nan', 'Age_18-24 years old', 'Age_25-

,__text__,MaxAge,AgeNum,WorkExp,YearsCode,RemoteCategoryNum,CompTotal,ConvertedCompYearly,ConvertedCompTotal,MainBranch_i am a developer by profession,...,"AISelect_yes, i use ai tools monthly or infrequently","AISelect_yes, i use ai tools weekly",AISelect_nan,"AIAgents_no, and i don't plan to","AIAgents_no, but i plan to","AIAgents_no, i use ai exclusively in copilot/autocomplete mode","AIAgents_yes, i use ai agents at work daily","AIAgents_yes, i use ai agents at work monthly or infrequently","AIAgents_yes, i use ai agents at work weekly",AIAgents_nan
0,"['bash/shell (all shells)', 'dart', 'sql'] ['d...",34.0,29.0,8.0,14.0,0.00,52800.0,61256.0,61659.84,1,...,1,0,0,0,0,0,0,1,0,0
1,"['java'] ['java', 'python', 'swift'] ['dynamod...",34.0,29.0,2.0,10.0,0.25,90000.0,104413.0,105102.00,1,...,0,1,0,1,0,0,0,0,0,0
3,"['java', 'kotlin', 'sql'] ['java', 'kotlin'] [...",44.0,39.0,4.0,5.0,0.00,31200.0,36197.0,36435.36,1,...,0,1,0,0,0,0,0,1,0,0
7,"['bash/shell (all shells)', 'html/css', 'javas...",44.0,39.0,22.0,30.0,0.00,72000.0,72000.0,72000.00,1,...,0,0,0,0,1,0,0,0,0,0
8,"['java', 'python', 'scala'] ['scala'] ['amazon...",34.0,29.0,9.0,15.0,0.00,70000.0,70000.0,70000.00,1,...,1,0,0,0,0,0,0,0,0,1


## Train/Test Split



In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 12520
Test size: 3131
Train class distribution:
 RemoteCategoryNum
0    0.355990
3    0.199361
1    0.190176
2    0.130112
4    0.124361
Name: proportion, dtype: float64
Test class distribution:
 RemoteCategoryNum
0    0.356116
3    0.199617
1    0.190035
2    0.129990
4    0.124241
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten




In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

preprocessor

,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'


## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit XGBoost-Feature-Importances)
- Klassifikator (XGBClassifier)


In [17]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))),
    ("classifier", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

pipeline


,steps,"[('preprocessing', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## GridSearchCV



In [18]:
parameters = {

    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],


    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [4, 6],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8],
    "classifier__colsample_bytree": [0.8],
    "classifier__reg_lambda": [1.0, 2.0],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)

grid


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__colsample_bytree': [0.8], 'classifier__learning_rate': [0.05, 0.1], 'classifier__max_depth': [4, 6], 'classifier__n_estimators': [300, 500], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('text', ...), ('num', ...)]"


## Grid Search + Beste Parameter


In [19]:

grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=1.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=0.9, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=   8.0s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=1.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=0.9, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=   8.3s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=2.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=

## Evaluation auf Testdaten


In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2,3,4]))

Classification Report (Test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1115
           1       1.00      1.00      1.00       595
           2       1.00      1.00      1.00       407
           3       1.00      1.00      1.00       625
           4       1.00      1.00      1.00       389

    accuracy                           1.00      3131
   macro avg       1.00      1.00      1.00      3131
weighted avg       1.00      1.00      1.00      3131

Confusion Matrix (rows=true, cols=pred):
[[1115    0    0    0]
 [   0  595    0    0]
 [   0    0  407    0]
 [   0    0    0  625]]


In [22]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4457
           1       1.00      1.00      1.00      2381
           2       1.00      1.00      1.00      1629
           3       1.00      1.00      1.00      2496
           4       1.00      1.00      1.00      1557

    accuracy                           1.00     12520
   macro avg       1.00      1.00      1.00     12520
weighted avg       1.00      1.00      1.00     12520

